# Example 3c: Beam from wavefront and apply a complex wavefront

**Goal.** Reconstruct a ray-based beam from a complex wavefront (intensity + phase), then apply a second complex wavefront to the same rays.

The first step follows the same inverse wavefront-to-ray mapping as Example 3b: ray positions are sampled from the wavefront intensity, while local ray directions are obtained from phase gradients:

    dX = (1 / k) * dphi/dx
    dY = (1 / k) * dphi/dy

with `k = 2*pi / wavelength`.

The second step uses `Beam.apply_wavefront()` to propagate an additional complex wavefront contribution, such as optical errors from a CRL.

Reference for the speckle-tracking principle used by this inverse mapping:
Celestre et al., *J. Synchrotron Rad.* **32**, 180–199 (2025).

In [ ]:
__author__ = ['Rafael Celestre']
__contact__ = 'rafael.celestre@esrf.eu'
__license__ = 'CECILL-2.1'
__copyright__ = 'ESRF - the European Synchrotron, Grenoble, France'
__created__ = '2025.05.01'
__changed__ = '2026.05.04'

import h5py
import matplotlib.pyplot as plt

import barc4beams as b4b

## 0) User parameters

In [ ]:
N_RAYS    = 50000
THRESHOLD = 1E-4
JITTER    = True
SEED      = 42
Z0        = 0.0

## 1) Load SRW intensity maps (HDF5)
We need:
- **far_field** dictionary with `intensity`; `x_axis`  and `y_axis` in (rad)
- **near_field** dictionary with `intensity`; `x_axis` and  `y_axis` in (m)
- the attribute `energy`

In [ ]:
path = "./beams/srw_crl_focusing_wavefront.h5"

with h5py.File(path, "r") as f:
    energy = float(f["energy_eV"][()])

    wavefront = {
        k: f["wavefront"][k][()]
        for k in ("intensity", "phase", "x_axis", "y_axis")
    }

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

im0 = axes[0].pcolormesh(
    wavefront["x_axis"] * 1e6,
    wavefront["y_axis"] * 1e6,
    wavefront["intensity"],
    shading="auto",
    cmap="turbo",
)

axes[0].set_title("Wavefront intensity")
axes[0].set_xlabel("x [µm]")
axes[0].set_ylabel("y [µm]")
axes[0].set_aspect("equal")

fig.colorbar(im0, ax=axes[0], label="Intensity [a.u.]")

im1 = axes[1].pcolormesh(
    wavefront["x_axis"] * 1e6,
    wavefront["y_axis"] * 1e6,
    wavefront["phase"],
    shading="auto",
    cmap="viridis",
)

axes[1].set_title("Wavefront phase")
axes[1].set_xlabel("x [µm]")
axes[1].set_ylabel("y [µm]")
axes[1].set_aspect("equal")

fig.colorbar(im1, ax=axes[1], label="Phase [rad]")

plt.show()

## 2) Reconstruct a beam from intensity and phase

Ray positions are sampled from the intensity map. Ray directions are obtained from the phase gradients, i.e. an inverse wavefront-to-ray mapping.

Unlike Example 3a, this reconstruction uses phase information. This provides local ray directions directly from the complex wavefront.

In [ ]:
beam_from_wft = b4b.Beam.from_wavefront(
    wavefront=wavefront,
    n_rays=N_RAYS,
    energy=energy,
    jitter=JITTER,
    threshold=THRESHOLD,
    seed=SEED,
    z0=Z0,
    polarization_degree=1.0,
)


## 3) Analyze the reconstructed beam

The reconstructed beam can now be analyzed with the same tools used for ray-tracing outputs.


In [ ]:
stats_ref = beam_from_wft.stats(verbose=True)
stats_ref


In [ ]:
plot = beam_from_wft.plot_beam(mode="hist2d", aspect_ratio=True, color=3, path=None, bins=100)
plot = beam_from_wft.plot_divergence(mode="hist2d", aspect_ratio=True, color=3, path=None, bins=100)


In [ ]:
plot = beam_from_wft.plot_caustic(
    which="both",
    n_points=101,
    xy_range=[-5, 5],
    start=stats_ref["fx"][0] - 0.1,
    finish=stats_ref["fx"][0] + 0.1,
    color=3,
)


## 4) Load an additional complex wavefront

Here we load a second wavefront representing the transmitted wavefront after optical errors, for example CRL figure or thickness errors.

This wavefront will be applied to the previously reconstructed beam.


In [ ]:
path = "./beams/srw_crl_transmitted_wavefront_err.h5"

with h5py.File(path, "r") as f:
    energy_err = float(f["energy_eV"][()])

    wavefront_err = {
        k: f["wavefront"][k][()]
        for k in ("intensity", "phase", "x_axis", "y_axis")
    }


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

im0 = axes[0].pcolormesh(
    wavefront_err["x_axis"] * 1e6,
    wavefront_err["y_axis"] * 1e6,
    wavefront_err["intensity"],
    shading="auto",
    cmap="turbo",
)

axes[0].set_title("Wavefront intensity")
axes[0].set_xlabel("x [µm]")
axes[0].set_ylabel("y [µm]")
axes[0].set_aspect("equal")

fig.colorbar(im0, ax=axes[0], label="Intensity [a.u.]")

im1 = axes[1].pcolormesh(
    wavefront_err["x_axis"] * 1e6,
    wavefront_err["y_axis"] * 1e6,
    wavefront_err["phase"],
    shading="auto",
    cmap="viridis",
)

axes[1].set_title("Wavefront phase")
axes[1].set_xlabel("x [µm]")
axes[1].set_ylabel("y [µm]")
axes[1].set_aspect("equal")

fig.colorbar(im1, ax=axes[1], label="Phase [rad]")

plt.show()

## 5) Apply the complex wavefront to the beam

`Beam.apply_wavefront()` updates the beam using the supplied complex wavefront:

- the intensity is modified according to the wavefront transmission,
- the ray directions are modified according to the local phase gradients,
- rays may be flagged as lost depending on the intensity threshold.

This provides a compact hybrid workflow: wavefront information is injected into a ray-based beam.


In [ ]:
aberrated_beam = beam_from_wft.apply_wavefront(
    wavefront=wavefront_err,
    energy=energy_err,
    threshold=THRESHOLD,
)

## 6) Analyze the aberrated beam

The aberrated beam is again a standard `Beam` object, so the same statistics and plotting methods apply.


In [ ]:
stats_err = aberrated_beam.stats(verbose=True)
stats_err

In [ ]:
plot = aberrated_beam.plot_beam(mode="hist2d", aspect_ratio=True, color=3, path=None, bins=100)
plot = aberrated_beam.plot_divergence(mode="hist2d", aspect_ratio=True, color=3, path=None, bins=100)


In [ ]:
plot = aberrated_beam.plot_caustic(
    which="both",
    n_points=101,
    xy_range=[-5, 5],
    start=stats_err["fx"][0] - 0.1,
    finish=stats_err["fx"][0] + 0.1,
    color=3,
)